# The Confusion Matrix

**Notebook 2 of 5.** Notebook 01 ended with two counts that accuracy had hidden: frauds
caught and frauds missed. This notebook completes the picture. Every prediction of a
two-class model lands in one of **four boxes**, and the table of those four counts, the
**confusion matrix**, is what every other classification metric is computed from.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

The same steps as in notebook 01, in one cell: load, split, scale, and train a logistic
regression.

In [ ]:
df = pd.read_csv("data/transactions.csv")
X = df.drop(columns="fraud")
y = df["fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000).fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

## 1. Four outcomes

A prediction is either right or wrong, and it said either fraud or honest. That makes four
combinations, each with a name. **Positive** and **negative** refer to what was
*predicted*; **true** and **false** to whether that was right.

| | Predicted honest (0) | Predicted fraud (1) |
| :-- | :-- | :-- |
| **Actually honest (0)** | **true negative (TN)**: left alone, correctly | **false positive (FP)**: an honest customer's card blocked |
| **Actually fraud (1)** | **false negative (FN)**: a fraud that got through | **true positive (TP)**: a fraud caught |

The two kinds of mistake are not alike. A false positive annoys a customer; a false
negative loses money.

> **Exercise:**
>
> Count the four outcomes for the model's test predictions, `y_pred` against `y_test`.

<details><summary>
Click here for a hint...
</summary>

Each box is a combination of a condition on the truth and one on the prediction. True
positives, for instance, are `((y_test == 1) & (y_pred == 1)).sum()`.
</details>

In [ ]:
# your code: TN, FP, FN and TP

<details><summary>
Click here to compare your answer...
</summary>

**12,374 true negatives, 3 false positives, 26 false negatives and 97 true positives.**
The four add up to the 12,500 test payments. Three honest customers had their cards
blocked, and 26 frauds got through.
</details>

## 2. The matrix in scikit-learn

`confusion_matrix` does the counting. Its layout is always the same: **rows are the
truth, columns are the prediction**, both in the order 0, 1.

In [ ]:
confusion_matrix(y_test, y_pred)

So the negatives sit in the top row and the positives in the bottom row, and the
correct predictions run along the diagonal. `ConfusionMatrixDisplay` draws the same table:

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["honest", "fraud"], cmap="Blues"
)
plt.title("The model on the test payments");

Look at the colors: the top left box is dark and the rest look alike, although 3 and 26
are very different numbers. On imbalanced data the majority class dominates the picture
too. Always read the numbers.

> **Exercise:**
>
> Draw the confusion matrix of the do-nothing baseline from notebook 01, which calls every
> payment honest. Which boxes are empty, and why?

In [ ]:
# your code: the confusion matrix of the baseline

<details><summary>
Click here to compare your answer...
</summary>

The whole **right column is empty**: the baseline never predicts fraud, so it has no true
positives and no false positives. All 123 frauds are false negatives. Its 99 percent
accuracy comes entirely from the top left box.
</details>

## 3. Which mistake costs more?

A model can usually trade one kind of mistake for the other, so the question that decides
how to judge it is: **which one is worse here?** There is no general answer; it depends on
the problem.

> **Exercise:**
>
> No code needed. For each problem, say what a false positive and a false negative are,
> and which one is worse.
>
> 1. Screening patients for a serious disease that is treatable if found early.
> 2. A spam filter for a work inbox.
> 3. Flagging card payments as fraud, as in this notebook.

Your answers here:

<details><summary>
Click here to compare your answers...
</summary>

1. **False negatives are worse.** A false positive sends a healthy person for a second
   test; a false negative sends a sick person home.
2. **False positives are worse.** A false negative leaves a spam email in the inbox; a
   false positive hides a real email, maybe an important one, in the spam folder.
3. **It is a balance.** A missed fraud costs money, but blocking honest customers costs
   goodwill and phone calls. Most banks lean toward catching fraud, and accept some false
   alarms.

The confusion matrix does not decide this. It only makes the two mistakes visible, so that
the decision can be made on purpose.
</details>

## Summary

- Every prediction is a **TP, FP, FN or TN**: *positive* or *negative* is what was
  predicted, *true* or *false* whether it was right.
- `confusion_matrix(y_true, y_pred)` puts **the truth in rows and the prediction in
  columns**, negatives first.
- **Which mistake is worse depends on the problem**, and deciding it comes before choosing
  a metric.

Next, in `03_precision_recall_f_scores.ipynb`: turning the four counts into the numbers
that answer "how many frauds did we catch?" and "how many alarms were real?".